# 02 — Integración de dos modelos: $t_0=10$ s y $t_0=50$ s

Se integran dos simulaciones con la misma red `pynucastro` y la misma razón inicial $n/p=1/7$, pero con diferente tiempo inicial. Se guardan CSVs completos para la memoria y para las gráficas comparativas.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scienceplots
    plt.style.use(["science", "grid"])
except Exception as exc:
    print("scienceplots no disponible, usando estilo por defecto:", exc)

from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp
import importlib.util

BASE = Path.cwd()
FIG_DIR = BASE / "figures"
DATA_DIR = BASE / "data"
TAB_DIR = BASE / "tables"
for d in [FIG_DIR, DATA_DIR, TAB_DIR]:
    d.mkdir(exist_ok=True)

In [14]:
# Cargamos red generada en el notebook 01.
spec = importlib.util.spec_from_file_location("bbn_network", BASE / "bbn_network.py")
bbn = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bbn)

print("Núcleos:", bbn.names)
print("A:", bbn.A)
print("Z:", bbn.Z)
name_to_i = {name: i for i, name in enumerate(bbn.names)}

# Compatibilidad entre nombres de pynucastro:
# algunas versiones escriben p,d,t; otras h1,h2,h3.
ALIASES = {
    "n":   ("n",),
    "p":   ("p", "h1", "H1"),
    "d":   ("d", "h2", "H2"),
    "t":   ("t", "h3", "H3"),
    "he3": ("he3", "He3"),
    "he4": ("he4",  "He4"),
    "li7": ("li7", "Li7"),
    "be7": ("be7",  "Be7"),
}

def nuc_name(key):
    for candidate in ALIASES[key]:
        if candidate in name_to_i:
            return candidate
    raise KeyError(f"No encuentro el núcleo {key}. Nombres disponibles: {bbn.names}")

def nuc_index(key):
    return name_to_i[nuc_name(key)]

def nuc_col(prefix, key):
    return f"{prefix}_{nuc_name(key).replace(' ', '')}"

CANON = {key: nuc_name(key) for key in ALIASES}
print("Aliases usados:", CANON)

Núcleos: ['n', 'H1', 'H2', 'H3', 'He3', 'He4', 'Li7', 'Be7']
A: [1 1 2 3 3 4 7 7]
Z: [0 1 1 1 2 2 3 4]
Aliases usados: {'n': 'n', 'p': 'H1', 'd': 'H2', 't': 'H3', 'he3': 'He3', 'he4': 'He4', 'li7': 'Li7', 'be7': 'Be7'}


In [15]:
# Trayectoria termodinámica del notebook 00.
snapshots = pd.read_csv(DATA_DIR / "bbn_assignment_snapshots.csv")
_t_nodes = snapshots["t_s"].to_numpy(float)
_T9_nodes = snapshots["T9"].to_numpy(float)
_rho_nodes = snapshots["rho_g_cm3"].to_numpy(float)
_logT9 = interp1d(np.log(_t_nodes), np.log(_T9_nodes), fill_value="extrapolate")
_logrho = interp1d(np.log(_t_nodes), np.log(_rho_nodes), fill_value="extrapolate")

def T9_of_t(t): return np.exp(_logT9(np.log(np.asarray(t, dtype=float))))
def T_of_t(t): return 1.0e9 * T9_of_t(t)
def rho_of_t(t): return np.exp(_logrho(np.log(np.asarray(t, dtype=float))))

def n_gamma_cm3(T_K): return 20.28 * np.asarray(T_K, dtype=float)**3

def eta_of_t(t):
    m_u = 1.66053906660e-24  # g
    return (rho_of_t(t)/m_u) / n_gamma_cm3(T_of_t(t))

In [ ]:
# Funciones de integración y postprocesado.

def make_initial_Y(np_ratio=1.0/7.0):
    Y0 = np.zeros(bbn.nnuc)
    # np_ratio = n/p, con Y_n + Y_p = 1 para A=1.
    Yn = np_ratio / (1.0 + np_ratio)
    Yp = 1.0 / (1.0 + np_ratio)
    Y0[nuc_index("n")] = Yn
    Y0[nuc_index("p")] = Yp
    return Y0

def rhs_var(t, Y):
    Y_safe = np.maximum(np.asarray(Y, dtype=float), 0.0)
    return np.asarray(bbn.rhs(t, Y_safe, float(rho_of_t(t)), float(T_of_t(t))), dtype=float)

def run_model(t0, t_end=1.0e5, label="model"):
    # Incluimos siempre los snapshots pedidos y un mallado logarítmico denso.
    requested = np.array([50.0, 200.0, 1000.0])
    requested = requested[(requested >= t0) & (requested <= t_end)]
    t_eval = np.unique(np.concatenate([np.geomspace(t0, t_end, 700), requested]))
    Y0 = make_initial_Y(np_ratio=1.0/7.0)
    sol = solve_ivp(
        rhs_var, (t0, t_end), Y0,
        method="LSODA", t_eval=t_eval,
        rtol=1e-6, atol=1e-30, max_step=2.0,
    )
    if not sol.success:
        raise RuntimeError(f"Falló {label}: {sol.message}")
    return sol

def solution_to_dataframe(sol, model_label, t0):
    Y_hist = np.maximum(sol.y.T, 0.0)
    X_hist = Y_hist * bbn.A[np.newaxis, :]
    df = pd.DataFrame({
        "model": model_label,
        "t0_s": t0,
        "t_s": sol.t,
        "T9": T9_of_t(sol.t),
        "T_K": T_of_t(sol.t),
        "rho_g_cm3": rho_of_t(sol.t),
        "eta": eta_of_t(sol.t),
        "eta10": 1.0e10 * eta_of_t(sol.t),
        "baryon_sum": np.sum(X_hist, axis=1),
    })
    for i, name in enumerate(bbn.names):
        clean = name.replace(" ", "")
        df[f"Y_{clean}"] = Y_hist[:, i]
        df[f"X_{clean}"] = X_hist[:, i]
    return df

def abundance_ratios(df):
    out = df.copy()
    Yp = out[nuc_col("Y", "p")].replace(0, np.nan)

    def ratio(key):
        col = nuc_col("Y", key)
        return out[col] / Yp if col in out.columns else np.nan

    out["D/H"] = ratio("d")
    out["T/H"] = ratio("t")
    out["He3/H"] = ratio("he3")
    out["Li7/H"] = ratio("li7")
    out["Be7/H"] = ratio("be7")
    out["Li7_plus_Be7_over_H"] = out["Li7/H"].fillna(0) + out["Be7/H"].fillna(0)
    out["X_He4"] = out[nuc_col("X", "he4")]
    out["X_H1"] = out[nuc_col("X", "p")]
    return out

def snapshot_table(df, snapshot_times=(50.0, 200.0, 1000.0)):
    rows = []
    for t in snapshot_times:
        if t < df["t_s"].min() or t > df["t_s"].max():
            continue
        row = {"model": df["model"].iloc[0], "t0_s": df["t0_s"].iloc[0], "t_s": t}
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            row[col] = float(np.interp(t, df["t_s"], df[col]))
        rows.append(row)
    return pd.DataFrame(rows)

In [17]:
# Ejecutamos las dos simulaciones.
sol10 = run_model(10.0, label="t0_10s")
sol50 = run_model(50.0, label="t0_50s")

model10 = abundance_ratios(solution_to_dataframe(sol10, "t0_10s", 10.0))
model50 = abundance_ratios(solution_to_dataframe(sol50, "t0_50s", 50.0))

model10.to_csv(DATA_DIR / "bbn_evolution_t0_10s.csv", index=False)
model50.to_csv(DATA_DIR / "bbn_evolution_t0_50s.csv", index=False)

all_models = pd.concat([model10, model50], ignore_index=True)
all_models.to_csv(DATA_DIR / "bbn_evolution_all_models.csv", index=False)

snap10 = snapshot_table(model10)
snap50 = snapshot_table(model50)
snap_all = pd.concat([snap10, snap50], ignore_index=True)
snap_all.to_csv(DATA_DIR / "bbn_snapshots_all_models.csv", index=False)

# Finales al final de la integración
final_all = all_models.sort_values("t_s").groupby("model", as_index=False).tail(1)
final_all.to_csv(DATA_DIR / "bbn_final_abundances_all_models.csv", index=False)

snap_all[["model", "t_s", "T9", "rho_g_cm3", "eta10", "D/H", "He3/H", "X_He4", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H", "baryon_sum"]]

 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.1122379272162D-27
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.1122379272162D-27
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.1122379272162D-23
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.1122379272162D-23
 lsoda--  warning..internal t (=r1) 

,model,t_s,T9,rho_g_cm3,eta10,D/H,He3/H,X_He4,Li7/H,Be7/H,Li7_plus_Be7_over_H,baryon_sum
0,t0_10s,50.0,1.7,0.000200,12.088327,2.967373e-19,4.752832e-06,2.499796e-01,2.486241e-21,1.530876e-06,1.530876e-06,1.0
1,t0_10s,200.0,1.0,0.000020,5.938995,2.943857e-19,4.554095e-06,2.499795e-01,6.553614e-21,1.614217e-06,1.614217e-06,1.0
2,t0_10s,1000.0,0.4,0.000002,9.279679,3.546831e-19,4.543060e-06,2.499795e-01,7.141584e-20,1.619403e-06,1.619403e-06,1.0
3,t0_50s,50.0,1.7,0.000200,12.088327,1.142857e-27,2.402034e-55,1.614236e-84,0.000000e+00,0.000000e+00,0.000000e+00,1.0
4,t0_50s,200.0,1.0,0.000020,5.938995,2.692124e-08,5.225594e-06,2.498882e-01,4.254565e-16,8.403180e-08,8.403180e-08,1.0
5,t0_50s,1000.0,0.4,0.000002,9.279679,5.673982e-09,5.209301e-06,2.498882e-01,4.719510e-17,9.008675e-08,9.008675e-08,1.0


In [18]:
# Tabla resumen final con sólo los observables más relevantes.
obs_cols = ["model", "t0_s", "t_s", "T9", "rho_g_cm3", "eta10", "D/H", "He3/H", "X_He4", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H", "baryon_sum"]
final_compact = final_all[obs_cols].copy()
final_compact.to_csv(DATA_DIR / "bbn_final_compact.csv", index=False)
final_compact

,model,t0_s,t_s,T9,rho_g_cm3,eta10,D/H,He3/H,X_He4,Li7/H,Be7/H,Li7_plus_Be7_over_H,baryon_sum
1404,t0_50s,50.0,100000.0,0.029068,2.752180e-09,33.275187,4.979074e-09,0.000005,0.249888,1.017326e-16,9.013511e-08,9.013511e-08,1.0
702,t0_10s,10.0,100000.0,0.029068,2.752180e-09,33.275187,4.013645e-19,0.000005,0.249979,2.694339e-18,1.619445e-06,1.619445e-06,1.0
